# Genres

- [Create Complete List of All Unique Genres](#create-complete-list-of-all-unique-genres)
- [Add Indexes to Final List of Artists and Genres](#add-indexes-to-final-list-of-artists-and-genres)
- [Genre Hierarchy](#genre-hierarchy)
- [Write Data to Excel File](#write-data-to-excel-file)

The final aspect of the Sounds of San Anto data that needs to be looked at is musical genre. 

Through 2024 and 2025, an effort was made to connect artists to genres using API calls to Wikipedia and Spotify. This resulted in 1769 artists having genres associated to them. Most artists had multiple genres. Each individual genre also had a genre index. There were 985 unique genres indexed.

Then, in 2025, librarian Jacob Sherman manually associated another 4618 artists to genres. 

So far, all the categories of data (artists, venues, concerts) have unique indexes, so we should continue that policy with genres. Accordingly, I will make sure any genres in Jacob Sherman's list are in the master genre list and have indexes. I will then create a list of all unique genres in our data.

I'll start by loading and inspecting the relevant spreadsheets.

In [1]:
import pandas as pd
import os

In [2]:
orig_genre_list = pd.read_excel('input-data/product1_data_master_rev.xlsx', sheet_name='genreIndex')

In [3]:
orig_genre_list.info()

<class 'pandas.DataFrame'>
RangeIndex: 985 entries, 0 to 984
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Genres  985 non-null    str  
 1   Index   985 non-null    int64
dtypes: int64(1), str(1)
memory usage: 15.5 KB


In [4]:
new_artists_genres = pd.read_excel('input-data/artists_genres_new.xlsx')

In [5]:
new_artists_genres.info()

<class 'pandas.DataFrame'>
RangeIndex: 10102 entries, 0 to 10101
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Name                     10101 non-null  str    
 1   Sheet1.name variation    261 non-null    str    
 2   Genres                   10096 non-null  str    
 3   artistIndex              10102 non-null  int64  
 4   Artists_Genre (2).Index  4791 non-null   float64
 5   Name2                    5305 non-null   str    
dtypes: float64(1), int64(1), str(4)
memory usage: 473.7 KB


In [6]:
new_artists_genres = new_artists_genres.rename(columns={'Artists_Genre (2).Index': 'Index'})

In [7]:
new_artists_genres.info()

<class 'pandas.DataFrame'>
RangeIndex: 10102 entries, 0 to 10101
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Name                   10101 non-null  str    
 1   Sheet1.name variation  261 non-null    str    
 2   Genres                 10096 non-null  str    
 3   artistIndex            10102 non-null  int64  
 4   Index                  4791 non-null   float64
 5   Name2                  5305 non-null   str    
dtypes: float64(1), int64(1), str(4)
memory usage: 473.7 KB


## Create Complete List of All Unique Genres

The first thing I'd like to do is ensure that all the genres from `new_artists_genres` are in the master genre list and all have a unique index. 

I use set() to make a list of all the genres in the original list. This list will be used only for lookup purposes, to see if a genre from the updated list is new or not. I check its length and find it has 985 genres.

Then I create a list of the genres in the second list using `.unique()`. Using this on a Pandas data frame generates an iterable numpy array, which retains the original order. I calculate its length and find that there are 1124 genres. This means we do have 139 new genres to add to the master list.

In [15]:
orig_genres_copy = orig_genre_list.copy()
orig_genres_list_lc = set(orig_genre_list['Genres'].str.lower())
len(orig_genres_list_lc)

985

In [16]:
new_genres_list = new_artists_genres['Genres'].dropna().unique()
len(new_genres_list)

1124

I'll go through all the new genres, doing a case-insensitive check. If they are not in the original list I'll add them to a `new_genres` list. I see that when I do a case-insensitive check there are only 60 new genres, so a lot of the difference between the lists is down to lower-case / upper-case differences.

In [17]:
already_checked = set()
new_genres = []
for genre in new_genres_list:
    genre_lc = genre.lower()
    if genre_lc not in orig_genres_list_lc and genre_lc not in already_checked:
        new_genres.append(genre)
        already_checked.add(genre_lc)

len(new_genres)

60

Now I'll add the new genres to the master list and give them unique indexes. Then I'll put the Index column first, as I've been doing for the other categories of data. I'll also rename the "Genres" column "Genre" since each row contains exactly one genre.

In [18]:
idx_start = orig_genres_copy['Index'].max() + 1
new_rows = pd.DataFrame({
    'Genres': new_genres,
    'Index': range(idx_start, idx_start + len(new_genres))
})
master_genres_list = pd.concat([orig_genre_list, new_rows], ignore_index=True)
master_genres_list.info()

<class 'pandas.DataFrame'>
RangeIndex: 1045 entries, 0 to 1044
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Genres  1045 non-null   str  
 1   Index   1045 non-null   int64
dtypes: int64(1), str(1)
memory usage: 16.5 KB


In [19]:
master_genres_list.head()

,Genres,Index
0,Contemporary Country,1
1,Country,2
2,Country Road,3
3,Unknown,4
4,Australian Country,5


In [22]:
master_genres_list = master_genres_list[['Index', 'Genres']]
master_genres_list = master_genres_list.rename(columns={'Genres': 'Genre'})
master_genres_list.info()

<class 'pandas.DataFrame'>
RangeIndex: 1045 entries, 0 to 1044
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Index   1045 non-null   int64
 1   Genre   1045 non-null   str  
dtypes: int64(1), str(1)
memory usage: 16.5 KB


## Add Indexes to Final List of Artists and Genres

Next, I would like it if all the genre associations in the new artist-genre list had genre indexes. All the artists had indexes added, but not all the genres. I'll create a copy to work with. Then I will use the master genre list to create a lookup dictionary for genre indexes. Then I'll use the `.map()` method to map the indexes to the corresponding genres wherever an index is missing. 

In [23]:
new_artists_genres_copy = new_artists_genres.copy()

genre_index_lookup = dict(zip(
    master_genres_list['Genre'].str.lower(),
    master_genres_list['Index']
))

missing_indexes = new_artists_genres_copy['Index'].isna()

new_artists_genres_copy.loc[missing_indexes, 'Index'] = (
    new_artists_genres_copy.loc[missing_indexes, 'Genres']
    .str.lower()
    .map(genre_index_lookup)
)

new_artists_genres_copy.info()

<class 'pandas.DataFrame'>
RangeIndex: 10102 entries, 0 to 10101
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Name                   10101 non-null  str    
 1   Sheet1.name variation  261 non-null    str    
 2   Genres                 10096 non-null  str    
 3   artistIndex            10102 non-null  int64  
 4   Index                  10096 non-null  float64
 5   Name2                  5305 non-null   str    
dtypes: float64(1), int64(1), str(4)
memory usage: 473.7 KB


I see that there are six artists that don't have genres. These should be dropped from the table.

In [25]:
new_artists_genres_copy[new_artists_genres_copy['Index'].isna()]

,Name,Sheet1.name variation,Genres,artistIndex,Index,Name2
399,Don Santiago & Flaco Jimenez,"Two separate artists, brothers",NaN,137,NaN,NaN
1802,Marcus Rubio,"mari maurice, More eaze",NaN,644,NaN,NaN
1971,Sara Jordan Powell,NaN,NaN,721,NaN,NaN
2572,The Snags,NaN,NaN,956,NaN,NaN
4159,Texas Trash,NaN,NaN,1505,NaN,NaN
4758,Sonny Ace,aka Sonny Ace and the Twisters,NaN,1746,NaN,NaN


In [26]:
master_artist_genre_list = new_artists_genres_copy.dropna(subset=['Index'])

In [27]:
master_artist_genre_list.info()

<class 'pandas.DataFrame'>
Index: 10096 entries, 0 to 10101
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Name                   10095 non-null  str    
 1   Sheet1.name variation  258 non-null    str    
 2   Genres                 10096 non-null  str    
 3   artistIndex            10096 non-null  int64  
 4   Index                  10096 non-null  float64
 5   Name2                  5305 non-null   str    
dtypes: float64(1), int64(1), str(4)
memory usage: 552.1 KB


Finally, I want a table that just includes artists, genres, and corresponding indexes.

In [28]:
master_artist_genre_list = master_artist_genre_list.drop(columns=['Sheet1.name variation', 'Name2'])

In [29]:
master_artist_genre_list = master_artist_genre_list.rename(columns={'Name': 'Artist', 'Genres': 'Genre', 'Index': 'genreIndex'})
master_artist_genre_list.info()

<class 'pandas.DataFrame'>
Index: 10096 entries, 0 to 10101
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Artist       10095 non-null  str    
 1   Genre        10096 non-null  str    
 2   artistIndex  10096 non-null  int64  
 3   genreIndex   10096 non-null  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 394.4 KB


We have a row with no artist. We'll drop that, too.

In [31]:
master_artist_genre_list = master_artist_genre_list.dropna(subset=['Artist'])
master_artist_genre_list.info()

<class 'pandas.DataFrame'>
Index: 10095 entries, 0 to 10101
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Artist       10095 non-null  str    
 1   Genre        10095 non-null  str    
 2   artistIndex  10095 non-null  int64  
 3   genreIndex   10095 non-null  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 394.3 KB


## Genre Hierarchy

There is one more bit of data that is potentially useful, that we generated in the course of working on Sounds of San Anto. Veronica Rodriguez, Diane Lopez, and I manually generated a list of ten basic genres, which we used in the Sounds of San Anto genres explorer. Each genre has a corresponding list of subgenres. We used this to group artists by genre for presentation purposes. 

I'll load the JSON file. Then I'll *normalize* it, which will expand the dictionary values into Pandas table columns. Finally, I'll use *explode*, which will take the index and genre values for each subgenre and fill in all the rows. This will make a nice presentation for sharing as an Excel spreadsheet.

In [32]:
genre_hierarchy = pd.read_json('input-data/genre_hierarchy_5.json')
genre_hierarchy.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 1 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   genres_and_subgenres  10 non-null     object
dtypes: object(1)
memory usage: 212.0+ bytes


In [33]:
genre_hierarchy

,genres_and_subgenres
0,"{'id': 1, 'genre': 'rock/pop', 'subgenres': ['..."
1,"{'id': 2, 'genre': 'country', 'subgenres': ['c..."
2,"{'id': 3, 'genre': 'tejano/conjunto', 'subgenr..."
3,"{'id': 4, 'genre': 'jazz', 'subgenres': ['big ..."
4,"{'id': 5, 'genre': 'r&b/rhythm & blues', 'subg..."
5,"{'id': 6, 'genre': 'metal', 'subgenres': ['uk ..."
6,"{'id': 7, 'genre': 'hip hop', 'subgenres': ['p..."
7,"{'id': 8, 'genre': 'latin', 'subgenres': ['lat..."
8,"{'id': 9, 'genre': 'electronic', 'subgenres': ..."
9,"{'id': 10, 'genre': 'regional mexican', 'subge..."


In [34]:
genre_hierarchy = pd.json_normalize(genre_hierarchy['genres_and_subgenres'])
genre_hierarchy = genre_hierarchy.explode('subgenres').reset_index(drop=True)
genre_hierarchy.info()

<class 'pandas.DataFrame'>
RangeIndex: 1081 entries, 0 to 1080
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   id         1081 non-null   int64
 1   genre      1081 non-null   str  
 2   subgenres  1081 non-null   str  
dtypes: int64(1), str(2)
memory usage: 25.5 KB


In [35]:
genre_hierarchy.head()

,id,genre,subgenres
0,1,rock/pop,nigerian pop
1,1,rock/pop,deep psychobilly
2,1,rock/pop,bedroom pop
3,1,rock/pop,neo-psychedelia
4,1,rock/pop,hardcore punk


## Write Data to Excel File

I've generated three tables worth sharing: a list of unique genres, a list of artists and genres, and a genre hierarchy. I'll write these three tables to our Excel file.

In [36]:
def write_df_to_excel(df, filepath, sheet_name):
    """Write a DataFrame to a sheet in an Excel file, creating the file
    if it doesn't exist, or replacing the sheet if it does."""
    
    mode = "a" if os.path.exists(filepath) else "w"
    kwargs = {"if_sheet_exists": "replace"} if mode == "a" else {}

    with pd.ExcelWriter(filepath, engine="openpyxl", datetime_format='YYYY-MM-DD', mode=mode, **kwargs) as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=False)

In [40]:
# write_df_to_excel(master_genres_list, 'output-data/sounds-of-san-anto.xlsx', 'genres')

In [41]:
# write_df_to_excel(master_artist_genre_list, 'output-data/sounds-of-san-anto.xlsx', 'artists_and_genres')

In [42]:
# write_df_to_excel(genre_hierarchy, 'output-data/sounds-of-san-anto.xlsx', 'genres_and_subgenres')